In [3]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import geopandas as gpd
import duckdb

print("=" * 70)
print("Combining Wind Turbine Data with ACS + Tract Data")
print("=" * 70)

# ============================================================
# 1. LOAD BOTH DATASETS
# ============================================================

# Load the cleaned ACS + tract data
tracts_acs = gpd.read_parquet("data/processed/tracts_with_acs_2024.parquet")
print(f"Tracts + ACS loaded: {len(tracts_acs):,} tracts")

# Load the filtered wind turbine data
turbines = gpd.read_parquet("data/processed/uswtdb_ia_ok_tx_2026.parquet")
print(f"Wind turbines loaded: {len(turbines):,} turbines")

# ============================================================
# 2. ENSURE BOTH DATASETS USE THE SAME CRS
# ============================================================

# Reproject turbines to match the tract CRS (important for spatial join)
if turbines.crs != tracts_acs.crs:
    turbines = turbines.to_crs(tracts_acs.crs)
    print(f"Turbines reprojected to: {tracts_acs.crs}")

# ============================================================
# 3. SPATIAL JOIN: Assign turbines to tracts
# ============================================================

# Perform spatial join (point-in-polygon)
turbines_in_tracts = gpd.sjoin(
    turbines, 
    tracts_acs[["GEOID", "geometry"]], 
    how="left", 
    predicate="within"
)

print(f"Turbines successfully joined to tracts: {len(turbines_in_tracts):,} records")

# ============================================================
# 4. AGGREGATE TURBINE STATISTICS PER TRACT
# ============================================================

turbine_stats = turbines_in_tracts.groupby("GEOID").agg(
    turbine_count=("GEOID", "count"),
    total_capacity_mw=("t_cap", "sum")   # t_cap = turbine capacity in MW
).reset_index()

# Create a binary flag for tracts that have at least one turbine
turbine_stats["has_wind_farm"] = (turbine_stats["turbine_count"] > 0).astype(int)

print(f"Tracts with turbines: {turbine_stats['turbine_count'].gt(0).sum():,}")

# ============================================================
# 5. MERGE TURBINE STATS BACK TO TRACTS + ACS
# ============================================================

final_data = tracts_acs.merge(
    turbine_stats, 
    on="GEOID", 
    how="left"
)

# Fill NaN values for tracts with no turbines
final_data["turbine_count"] = final_data["turbine_count"].fillna(0).astype(int)
final_data["total_capacity_mw"] = final_data["total_capacity_mw"].fillna(0)
final_data["has_wind_farm"] = final_data["has_wind_farm"].fillna(0).astype(int)

print(f"Final combined dataset shape: {final_data.shape}")

# ============================================================
# 6. SAVE THE COMBINED DATASET
# ============================================================

output_path = "data/processed/tracts_acs_wind_2024.parquet"
final_data.to_parquet(output_path, index=False)

print(f"✅ Combined dataset saved to: {output_path}")
print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Combining Wind Turbine Data with ACS + Tract Data
Tracts + ACS loaded: 8,985 tracts
Wind turbines loaded: 31,550 turbines
Turbines reprojected to: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Geodesy.", "area": "North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Sc

In [4]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

file_path = "data/processed/tracts_acs_wind_2024.parquet"

print("=" * 70)
print("Inspecting Combined Dataset: tracts_acs_wind_2024.parquet")
print("=" * 70)

# ============================================================
# 1. OVERALL SUMMARY
# ============================================================

print("\n--- 1. Overall Summary ---")
con.sql(f"""
    SELECT 
        COUNT(*) AS total_tracts,
        SUM(has_wind_farm) AS tracts_with_wind_farms,
        ROUND(100.0 * SUM(has_wind_farm) / COUNT(*), 2) AS pct_with_wind_farms,
        SUM(turbine_count) AS total_turbines,
        ROUND(AVG(turbine_count), 2) AS avg_turbines_per_tract
    FROM '{file_path}'
""").show()

# ============================================================
# 2. COMPARE SOCIO-ECONOMIC VARIABLES: With vs Without Wind Farms
# ============================================================

print("\n--- 2. Socio-Economic Comparison (With vs Without Wind Farms) ---")
con.sql(f"""
    SELECT 
        has_wind_farm,
        COUNT(*) AS tract_count,
        ROUND(AVG(total_population), 0) AS avg_population,
        ROUND(AVG(total_housing_units), 0) AS avg_housing_units,
        ROUND(AVG(median_household_income), 0) AS avg_income,
        ROUND(AVG(median_home_value), 0) AS avg_home_value
    FROM '{file_path}'
    GROUP BY has_wind_farm
    ORDER BY has_wind_farm
""").show()

# ============================================================
# 3. TOP TRACTS BY TURBINE COUNT
# ============================================================

print("\n--- 3. Top 10 Tracts by Turbine Count ---")
con.sql(f"""
    SELECT 
        GEOID,
        NAME,
        turbine_count,
        total_capacity_mw,
        total_population,
        median_household_income
    FROM '{file_path}'
    ORDER BY turbine_count DESC
    LIMIT 10
""").show()

# ============================================================
# 4. DISTRIBUTION OF TURBINE COUNT
# ============================================================

print("\n--- 4. Turbine Count Distribution ---")
con.sql(f"""
    SELECT 
        turbine_count,
        COUNT(*) AS tract_count
    FROM '{file_path}'
    GROUP BY turbine_count
    ORDER BY turbine_count
    LIMIT 15
""").show()

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Inspecting Combined Dataset: tracts_acs_wind_2024.parquet

--- 1. Overall Summary ---
┌──────────────┬────────────────────────┬─────────────────────┬────────────────┬────────────────────────┐
│ total_tracts │ tracts_with_wind_farms │ pct_with_wind_farms │ total_turbines │ avg_turbines_per_tract │
│    int64     │         int128         │       double        │     int128     │         double         │
├──────────────┼────────────────────────┼─────────────────────┼────────────────┼────────────────────────┤
│         8985 │                    374 │                4.16 │          31549 │                   3.51 │
└──────────────┴────────────────────────┴─────────────────────┴────────────────┴────────────────────────┘


--- 2. Socio-Economic Comparison (With vs Without Wind Farms) ---
┌───────────────┬─────────────┬────────────────┬───────────────────┬────────────